# 23CSE301 — Machine Learning Capstone — Review 1
## Regression Track — Battery State-of-Health

**Target:** predict **State-of-Health (SOH, %)**
of a battery cell — not raw Capacity directly. Source: NASA Battery Dataset
(Kaggle, `patrickfleith/nasa-battery-dataset`, `cleaned_dataset`) —
`metadata.csv` plus the ~7,000 per-cycle raw CSVs it references via its
`filename` column.

> **Reproducibility:** `random_state=42` throughout (§7.1).


## 0. Imports & configuration

In [4]:
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.autolayout"] = True

DATA_PATH = "../data/Regression_Dataset/metadata.csv"     # metadata file
RAW_DIR = "../data/Regression_Dataset/data/"              # per-cycle CSVs live here, named by metadata's `filename` column

NOMINAL_CAPACITY = 2.0   # Ah, rated capacity of these cells - used to convert Capacity -> SOH%


## 1. Load & audit the raw file (Section A1)

In [5]:
df_raw = pd.read_csv(DATA_PATH)
for col in ["Capacity", "Re", "Rct"]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

print("Shape:", df_raw.shape)
print("\nRow type counts:\n", df_raw["type"].value_counts())
print("\nUnique batteries:", df_raw["battery_id"].nunique())
print("\nMissing values per column:\n", df_raw.isna().sum())

Shape: (7565, 10)

Row type counts:
 type
charge       2815
discharge    2794
impedance    1956
Name: count, dtype: int64

Unique batteries: 34

Missing values per column:
 type                      0
start_time                0
ambient_temperature       0
battery_id                0
test_id                   0
uid                       0
filename                  0
Capacity               4796
Re                     5618
Rct                    5618
dtype: int64


**Observation (A1):** 

as before, missingness is structural :-
- `Capacity` only on `discharge` rows, `Re`/`Rct` only on `impedance` rows. 
- `charge` rows carry none of these but do have their own raw per-cycle curve.

### 1b. Structural join: one row per discharge cycle

In [9]:
def parse_start_time(s):
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s))
    if len(nums) < 6:
        return pd.NaT
    y, mo, d, h, mi, se = (float(n) for n in nums[:6])
    try:
        return pd.Timestamp(int(y), int(mo), int(d), int(h), int(mi), int(round(se)))
    except (ValueError, OverflowError):
        return pd.NaT

df_raw["parsed_time"] = df_raw["start_time"].apply(parse_start_time)

discharge = df_raw[df_raw["type"] == "discharge"].sort_values(["battery_id", "test_id"]).copy()
impedance = df_raw[df_raw["type"] == "impedance"].sort_values(["battery_id", "test_id"]).copy()
charge = df_raw[df_raw["type"] == "charge"].sort_values(["battery_id", "test_id"]).copy()

discharge["cycle_index"] = discharge.groupby("battery_id").cumcount() + 1

disc_slim = discharge[["battery_id", "test_id", "ambient_temperature", "cycle_index", "Capacity", "filename"]]
imp_slim = impedance[["battery_id", "test_id", "Re", "Rct"]]

analysis_df = pd.merge_asof(
    disc_slim.sort_values("test_id"), imp_slim.sort_values("test_id"),
    on="test_id", by="battery_id", direction="backward",
)
print("Per-discharge-cycle table:", analysis_df.shape)
analysis_df.head()


Per-discharge-cycle table: (2794, 8)


,battery_id,test_id,ambient_temperature,cycle_index,Capacity,filename,Re,Rct
0,B0034,0,24,1,0.745930,01805.csv,NaN,NaN
1,B0046,0,4,1,1.728239,00553.csv,NaN,NaN
2,B0041,0,4,1,0.055620,02996.csv,NaN,NaN
3,B0056,0,4,1,0.785278,07062.csv,NaN,NaN
4,B0040,0,24,1,0.673463,01231.csv,NaN,NaN
